In [2]:
from pathlib import Path

DATA_ROOT = Path.home() / "ecg_work"

def explore(path, max_depth=4, depth=0):
    if depth > max_depth:
        return
    indent = "  " * depth
    entries = sorted(path.iterdir())
    dirs = [e for e in entries if e.is_dir()]
    files = [e for e in entries if e.is_file()]
    for d in dirs:
        print(f"{indent}{d.name}/")
        explore(d, max_depth, depth + 1)
    for f in files[:5]:
        print(f"{indent}{f.name}")
    if len(files) > 5:
        print(f"{indent}... and {len(files) - 5} more files")

explore(DATA_ROOT)

.ipynb_checkpoints/
Child_ecg_extracted/
  Child_ecg/
    P00/
      P00001/
        P00001_E01.dat
        P00001_E01.hea
      P00002/
        P00002_E01.dat
        P00002_E01.hea
      P00003/
        P00003_E01.dat
        P00003_E01.hea
      P00004/
        P00004_E01.dat
        P00004_E01.hea
        P00004_E02.dat
        P00004_E02.hea
      P00005/
        P00005_E01.dat
        P00005_E01.hea
        P00005_E02.dat
        P00005_E02.hea
      P00006/
        P00006_E01.dat
        P00006_E01.hea
        P00006_E02.dat
        P00006_E02.hea
        P00006_E03.dat
        ... and 1 more files
      P00007/
        P00007_E01.dat
        P00007_E01.hea
      P00008/
        P00008_E01.dat
        P00008_E01.hea
        P00008_E02.dat
        P00008_E02.hea
        P00008_E03.dat
        ... and 3 more files
      P00009/
        P00009_E01.dat
        P00009_E01.hea
      P00010/
        P00010_E01.dat
        P00010_E01.hea
      P00011/
        P00011_E01.dat
        P000

In [21]:
import sys, os
sys.path.insert(0, os.path.expanduser("~/ecg_work"))

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import hamming_loss, f1_score

from data.dataset import (
    build_chen_labels, load_disease_code_table, window_level_split,
    ChenECGDataset, COL_N_LEADS,
)
from models.models import build_model
from utils.loss import compute_chen_class_weights, ChenWeightedBCELoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")

Device: cuda | Tesla V100S-PCIE-32GB


In [22]:
DATA_DIR = os.path.expanduser("~/ecg_work/Child_ecg_extracted/Child_ecg")
METADATA_CSV = os.path.expanduser("~/ecg_work/AttributesDictionary.csv")
DISEASE_CSV = os.path.expanduser("~/ecg_work/DiseaseCode.csv")

metadata = pd.read_csv(METADATA_CSV)
disease_table = load_disease_code_table(DISEASE_CSV)
labeled, label_columns = build_chen_labels(metadata, disease_table)
subset = labeled[labeled["in_chen_subset"]].reset_index(drop=True)

print("Chen subset:", len(subset), "(expect 3716)")
print("Label columns:", len(label_columns))
print(subset[COL_N_LEADS].value_counts())

Chen subset: 3716 (expect 3716)
Label columns: 16
Lead
12    2783
9      933
Name: count, dtype: int64


In [23]:
LEADS = 12
lead_subset = subset[subset[COL_N_LEADS] == LEADS].reset_index(drop=True)
print(f"{LEADS}-lead records:", len(lead_subset))

split = window_level_split(lead_subset, label_columns, DATA_DIR, seed=42)
print("Window split sizes:", {k: len(v) for k, v in split.items()})

12-lead records: 2783
Window split sizes: {'train': 18269, 'val': 2612, 'test': 5224}


In [24]:
train_ds = ChenECGDataset(lead_subset, label_columns, DATA_DIR, split["train"])
val_ds = ChenECGDataset(lead_subset, label_columns, DATA_DIR, split["val"])
test_ds = ChenECGDataset(lead_subset, label_columns, DATA_DIR, split["test"])

train_ds.fit_normalization()
val_ds.fit_normalization()
test_ds.fit_normalization()

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)

x0, y0 = train_ds[0]
print("Sample shapes:", x0.shape, y0.shape)

Sample shapes: torch.Size([12, 300]) torch.Size([16])


In [25]:
import time

model = build_model("resnet1d", n_leads=LEADS, n_classes=len(label_columns)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=7e-4)  # Chen's confirmed values

train_labels = lead_subset.iloc[[r for r, w in split["train"]]][label_columns].to_numpy()
class_weights = compute_chen_class_weights(train_labels).to(device)
criterion = ChenWeightedBCELoss(class_weights).to(device)

n_batches_total = len(train_loader)
print("Total batches per epoch:", n_batches_total)

n_test_batches = 10
start = time.time()
for i, (x, y) in enumerate(train_loader):
    if i >= n_test_batches:
        break
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()
elapsed = time.time() - start

per_batch = elapsed / n_test_batches
est_epoch_min = (per_batch * n_batches_total) / 60
print(f"{n_test_batches} batches: {elapsed:.1f}s ({per_batch:.3f}s/batch)")
print(f"Estimated full epoch: {est_epoch_min:.1f} minutes")

Total batches per epoch: 286
10 batches: 1.3s (0.126s/batch)
Estimated full epoch: 0.6 minutes


In [26]:
import torch
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

# rerun the exact same timing test
model = build_model("resnet1d", n_leads=LEADS, n_classes=len(label_columns)).to(device)
x_test, y_test = next(iter(train_loader))
x_test, y_test = x_test.to(device), y_test.to(device)
out = model(x_test)
print("Forward pass succeeded:", out.shape)

Forward pass succeeded: torch.Size([64, 16])


In [12]:
torch.backends.cudnn.enabled = False

In [14]:
torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True

model = build_model("resnet1d", n_leads=LEADS, n_classes=len(label_columns)).to(device)
x_test, y_test = next(iter(train_loader))
x_test, y_test = x_test.to(device), y_test.to(device)
out = model(x_test)
print("Forward pass succeeded:", out.shape)

RuntimeError: FIND was unable to find an engine to execute this computation

In [16]:
import torch
torch.backends.cudnn.enabled = False

model = build_model("resnet1d", n_leads=LEADS, n_classes=len(label_columns)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=7e-4)

n_batches_total = len(train_loader)
n_test_batches = 10
import time
start = time.time()
for i, (x, y) in enumerate(train_loader):
    if i >= n_test_batches:
        break
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad()
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()
elapsed = time.time() - start

per_batch = elapsed / n_test_batches
est_epoch_min = (per_batch * n_batches_total) / 60
print(f"{n_test_batches} batches: {elapsed:.1f}s ({per_batch:.3f}s/batch)")
print(f"Estimated full epoch: {est_epoch_min:.1f} minutes")

10 batches: 1.3s (0.133s/batch)
Estimated full epoch: 0.6 minutes


In [19]:
def evaluate(model, loader, device, threshold=0.5):
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            probs = torch.sigmoid(model(x))
            all_probs.append(probs.cpu().numpy())
            all_targets.append(y.numpy())
    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)
    y_pred = (y_prob >= threshold).astype(int)
    return hamming_loss(y_true, y_pred), f1_score(y_true, y_pred, average="macro", zero_division=0)


# fresh model -- the 10 batches from the timing test already nudged the old one's weights
model = build_model("resnet1d", n_leads=LEADS, n_classes=len(label_columns)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=7e-4)  # Chen's confirmed rate, no weight decay

N_EPOCHS = 15
best_f1 = -1.0
best_state = None
history = []

for epoch in range(N_EPOCHS):
    model.train()
    running_loss, n_batches = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        n_batches += 1
    train_loss = running_loss / n_batches

    val_hloss, val_f1 = evaluate(model, val_loader, device)
    history.append({"epoch": epoch+1, "train_loss": train_loss, "val_hamming": val_hloss, "val_f1": val_f1})

    marker = ""
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        marker = "  <- new best"
    print(f"Epoch {epoch+1}/{N_EPOCHS} - train_loss: {train_loss:.4f} - "
          f"val_hamming: {val_hloss:.4f} - val_macro_f1: {val_f1:.4f}{marker}")

model.load_state_dict(best_state)
print(f"\nBest val_macro_f1: {best_f1:.4f}")

Epoch 1/15 - train_loss: 1.0515 - val_hamming: 0.2735 - val_macro_f1: 0.1694  <- new best
Epoch 2/15 - train_loss: 0.8469 - val_hamming: 0.2466 - val_macro_f1: 0.2087  <- new best
Epoch 3/15 - train_loss: 0.7394 - val_hamming: 0.1801 - val_macro_f1: 0.2563  <- new best
Epoch 4/15 - train_loss: 0.6698 - val_hamming: 0.1958 - val_macro_f1: 0.2837  <- new best
Epoch 5/15 - train_loss: 0.6185 - val_hamming: 0.1782 - val_macro_f1: 0.2554
Epoch 6/15 - train_loss: 0.5748 - val_hamming: 0.1307 - val_macro_f1: 0.3056  <- new best
Epoch 7/15 - train_loss: 0.5260 - val_hamming: 0.1672 - val_macro_f1: 0.2699
Epoch 8/15 - train_loss: 0.4910 - val_hamming: 0.1591 - val_macro_f1: 0.3163  <- new best
Epoch 9/15 - train_loss: 0.4398 - val_hamming: 0.1356 - val_macro_f1: 0.3271  <- new best
Epoch 10/15 - train_loss: 0.4242 - val_hamming: 0.1137 - val_macro_f1: 0.3938  <- new best
Epoch 11/15 - train_loss: 0.4074 - val_hamming: 0.1283 - val_macro_f1: 0.4035  <- new best
Epoch 12/15 - train_loss: 0.3428 -

In [28]:
import torch
torch.backends.cudnn.enabled = False  # required every fresh session, not saved anywhere

model = build_model("resnet1d", n_leads=LEADS, n_classes=len(label_columns)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=7e-4)

N_EPOCHS = 200
best_f1 = -1.0
history = []
CHECKPOINT_PATH = os.path.expanduser("~/ecg_work/outputs/resnet1d_real_best.pt")

for epoch in range(N_EPOCHS):
    model.train()
    running_loss, n_batches = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        n_batches += 1
    train_loss = running_loss / n_batches

    val_hloss, val_f1 = evaluate(model, val_loader, device)
    history.append({"epoch": epoch+1, "train_loss": train_loss, "val_hamming": val_hloss, "val_f1": val_f1})

    marker = ""
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), CHECKPOINT_PATH)  # saved to disk immediately, not just RAM
        marker = "  <- new best, saved"
    print(f"Epoch {epoch+1}/{N_EPOCHS} - train_loss: {train_loss:.4f} - "
          f"val_hamming: {val_hloss:.4f} - val_macro_f1: {val_f1:.4f}{marker}")

print(f"\nTraining complete. Best val_macro_f1: {best_f1:.4f}")
model.load_state_dict(torch.load(CHECKPOINT_PATH))

Epoch 1/200 - train_loss: 1.0072 - val_hamming: 0.2333 - val_macro_f1: 0.1851  <- new best, saved
Epoch 2/200 - train_loss: 0.8198 - val_hamming: 0.2165 - val_macro_f1: 0.2123  <- new best, saved
Epoch 3/200 - train_loss: 0.7160 - val_hamming: 0.1979 - val_macro_f1: 0.2644  <- new best, saved
Epoch 4/200 - train_loss: 0.6376 - val_hamming: 0.1661 - val_macro_f1: 0.2664  <- new best, saved
Epoch 5/200 - train_loss: 0.6022 - val_hamming: 0.1355 - val_macro_f1: 0.3135  <- new best, saved
Epoch 6/200 - train_loss: 0.5462 - val_hamming: 0.1362 - val_macro_f1: 0.2932
Epoch 7/200 - train_loss: 0.4782 - val_hamming: 0.1281 - val_macro_f1: 0.3324  <- new best, saved
Epoch 8/200 - train_loss: 0.4744 - val_hamming: 0.1526 - val_macro_f1: 0.3101
Epoch 9/200 - train_loss: 0.4183 - val_hamming: 0.1219 - val_macro_f1: 0.3639  <- new best, saved
Epoch 10/200 - train_loss: 0.4058 - val_hamming: 0.1110 - val_macro_f1: 0.4225  <- new best, saved
Epoch 11/200 - train_loss: 0.3697 - val_hamming: 0.0919 - v

/tmp/ipykernel_113706/3218550571.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(CHECKPOINT_PATH))


<All keys matched successfully>

In [29]:
model.eval()
test_hloss, test_f1 = evaluate(model, test_loader, device)

print("ResNet-1D, 12-lead, FULL 200-epoch run, real hardware, real Chen parameters")
print(f"  Test macro F1:    {test_f1:.4f}   (Chen reported: 0.9467)")
print(f"  Test Hamming loss: {test_hloss:.4f}   (Chen reported: 0.0069)")
print(f"  Difference: {test_f1 - 0.9467:+.4f} F1, {test_hloss - 0.0069:+.4f} Hamming")

ResNet-1D, 12-lead, FULL 200-epoch run, real hardware, real Chen parameters
  Test macro F1:    0.9499   (Chen reported: 0.9467)
  Test Hamming loss: 0.0069   (Chen reported: 0.0069)
  Difference: +0.0032 F1, +0.0000 Hamming
